In [2]:
import os
os.getcwd()

'/Users/kjh/Documents/geostat3_e9'

## Find Unique $(i, t)$ Entities

In [5]:
import pandas as pd
from pathlib import Path

# 파일 경로
path = Path("/Users/kjh/Documents/geostat3_e9/E9_panel_sgg_aggregated.csv")  # 필요시 교체

# 1) 데이터 로드
df = pd.read_csv(path)

# 2) 컬럼명 소문자/공백 정리
df.columns = df.columns.str.strip().str.lower()

# 3) 후보 컬럼 자동 매핑
ALIAS = {
    "sido":    ["sido", "시도", "광역시도", "province", "sido_nm", "sido_name", "sido_name_kor"],
    "sigungu": ["sigungu", "시군구", "sgg", "시군구명", "sigungu_nm", "sigungu_name"],
    "year":    ["year", "연도", "yyyy"],
    "date":    ["date", "날짜", "ym", "yyyymm", "연월"]
}

def pick(col_aliases):
    for c in col_aliases:
        if c in df.columns:
            return c
    return None

sido_col    = pick(ALIAS["sido"])
sigungu_col = pick(ALIAS["sigungu"])
year_col    = pick(ALIAS["year"])
date_col    = pick(ALIAS["date"])  # 연도 파생용(없으면 None)

if sido_col is None or sigungu_col is None:
    raise KeyError("시도/시군구 컬럼을 찾지 못했습니다. 컬럼명을 확인해주세요.")

# 4) year 파생 (year가 없고 date/yyyymm만 있을 때)
if year_col is None:
    if date_col is None:
        raise KeyError("연도(year) 또는 연월(date/yyyymm) 관련 컬럼을 찾지 못했습니다.")
    # 문자열 변환 후 앞 4자리로 연도 추출
    year_series = pd.to_datetime(
        df[date_col].astype(str).str.replace(r"[^0-9]", "", regex=True).str[:6],  # 202001 같은 형태 유도
        format="%Y%m", errors="coerce"
    ).dt.year
else:
    # 숫자/문자 혼재 대비하여 깔끔한 연도로 정규화
    year_series = pd.to_numeric(df[year_col], errors="coerce")
    if year_series.isna().all() and date_col is not None:
        year_series = pd.to_datetime(
            df[date_col].astype(str).str.replace(r"[^0-9]", "", regex=True).str[:6],
            format="%Y%m", errors="coerce"
        ).dt.year

if year_series.isna().any():
    # 불가피할 때만 남김. 필요시 dropna()로 제거 가능
    pass

# 5) 유일 조합 테이블 생성
uniq = (
    pd.DataFrame({
        "sido": df[sido_col].astype(str).str.strip(),
        "sigungu": df[sigungu_col].astype(str).str.strip(),
        "year": year_series.astype("Int64")  # 결측 허용 정수
    })
    .dropna(subset=["sido", "sigungu", "year"])
    .drop_duplicates()
    .sort_values(["sido", "sigungu", "year"])
    .reset_index(drop=True)
)

# 6) 저장(선택)
uniq.to_csv("unique_e9.csv", index=False, encoding="utf-8-sig")

uniq.head()


,sido,sigungu,year
0,강원도,강릉시,2009
1,강원도,강릉시,2010
2,강원도,강릉시,2011
3,강원도,강릉시,2012
4,강원도,강릉시,2013


In [ ]:
import re
import pandas as pd
from pathlib import Path

# 파일 경로
path = Path("/Users/kjh/Documents/geostat3_e9/trade_panel.csv")

# 1) 데이터 로드
df = pd.read_csv(path)

# 2) 컬럼명 소문자/공백 정리
df.columns = df.columns.str.strip().str.lower()

# 3) 후보 컬럼 자동 매핑
ALIAS = {
    "sido":    ["sido", "시도", "광역시도", "province", "sido_nm", "sido_name", "sido_name_kor"],
    "sigungu": ["sigungu", "시군구", "sgg", "시군구명", "sigungu_nm", "sigungu_name"],
    "region":  ["region", "지역", "행정구역", "행정구역명", "시도_시군구"],
    "year":    ["year", "연도", "yyyy"],
    "date":    ["date", "날짜", "ym", "yyyymm", "연월"]
}

def pick(cols, in_df):
    for c in cols:
        if c in in_df.columns:
            return c
    return None

sido_col    = pick(ALIAS["sido"], df)
sigungu_col = pick(ALIAS["sigungu"], df)
region_col  = pick(ALIAS["region"], df)
year_col    = pick(ALIAS["year"], df)
date_col    = pick(ALIAS["date"], df)

# ---------- (A) region -> (sido, sigungu) 분리 ----------
# 공식/변경 포함 시도명 후보(긴 것 먼저 매칭)
SIDO_NAMES = [
    "세종특별자치시", "제주특별자치도", "강원특별자치도", "전북특별자치도",
    "서울특별시", "부산광역시", "대구광역시", "인천광역시", "광주광역시", "대전광역시", "울산광역시",
    "경기도", "강원도", "충청북도", "충청남도", "전라북도", "전라남도", "경상북도", "경상남도"
]
# 영문/축약 대응(있다면)
SIDO_EN_MAP = {
    "seoul": "서울특별시", "busan": "부산광역시", "daegu": "대구광역시", "incheon": "인천광역시",
    "gwangju": "광주광역시", "daejeon": "대전광역시", "ulsan": "울산광역시", "sejong": "세종특별자치시",
    "gyeonggi": "경기도", "gangwon": "강원특별자치도", "chungbuk": "충청북도", "chungnam": "충청남도",
    "jeonbuk": "전북특별자치도", "jeonnam": "전라남도", "gyeongbuk": "경상북도", "gyeongnam": "경상남도",
    "jeju": "제주특별자치도"
}

def clean_region(s):
    if pd.isna(s):
        return None
    s = str(s)
    s = re.sub(r"[\(\[\{].*?[\)\]\}]", "", s)  # 괄호 내용 제거
    s = s.replace("/", " ").replace("-", " ").replace(",", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_region(s):
    """ region 문자열에서 (시도, 시군구) 추출 """
    if not s:
        return (None, None)
    raw = clean_region(s)

    # 1) 쉼표/슬래시로 이미 구분된 케이스 처리 후 재시도
    for sep in [",", "/", "-"]:
        if sep in str(s):
            parts = [p.strip() for p in str(s).split(sep)]
            if len(parts) >= 2:
                return split_region(" ".join(parts))  # 공백 기반 재처리

    # 2) 시도명 prefix 매칭(가장 긴 시도명 우선)
    for sido_name in sorted(SIDO_NAMES, key=len, reverse=True):
        if raw.startswith(sido_name):
            rest = raw[len(sido_name):].strip()
            return (sido_name, rest if rest else None)

    # 3) 영문/로마자 시도명 시작하는 경우
    first_tok = raw.split(" ")[0].lower()
    if first_tok in SIDO_EN_MAP:
        sido_name = SIDO_EN_MAP[first_tok]
        rest = raw[len(first_tok):].strip()
        # 영문 토큰 제거를 위해 첫 토큰 제외
        rest = " ".join(raw.split(" ")[1:]).strip() or None
        return (sido_name, rest)

    # 4) fallback: 첫 토큰을 시도, 나머지를 시군구로
    toks = raw.split(" ")
    if len(toks) == 1:
        return (toks[0], None)
    return (toks[0], " ".join(toks[1:]))

# region이 있고(또는 시도/시군구 중 하나라도 없으면) region에서 파생
if region_col is not None and (sido_col is None or sigungu_col is None):
    tmp = df[region_col].map(clean_region)
    parsed = tmp.map(split_region)
    df["_sido_from_region"]    = parsed.map(lambda x: x[0])
    df["_sigungu_from_region"] = parsed.map(lambda x: x[1])
    # 우선순위: 기존 컬럼 > region 파생
    if sido_col is None:
        sido_col = "_sido_from_region"
    if sigungu_col is None:
        sigungu_col = "_sigungu_from_region"

# 최종 체크
if sido_col is None or sigungu_col is None:
    raise KeyError("시도/시군구 컬럼을 찾지 못했습니다. (region 분해 실패 가능)")

# ---------- (B) 연도 파생 ----------
if year_col is None:
    if date_col is None:
        raise KeyError("연도(year) 또는 연월(date/yyyymm) 관련 컬럼을 찾지 못했습니다.")
    year_series = pd.to_datetime(
        df[date_col].astype(str).str.replace(r"[^0-9]", "", regex=True).str[:6],
        format="%Y%m", errors="coerce"
    ).dt.year
else:
    year_series = pd.to_numeric(df[year_col], errors="coerce")
    if year_series.isna().all() and date_col is not None:
        year_series = pd.to_datetime(
            df[date_col].astype(str).str.replace(r"[^0-9]", "", regex=True).str[:6],
            format="%Y%m", errors="coerce"
        ).dt.year

# ---------- (C) 유일 조합 테이블 ----------
uniq = (
    pd.DataFrame({
        "sido": df[sido_col].astype(str).str.strip(),
        "sigungu": df[sigungu_col].astype(str).str.strip(),
        "year": year_series.astype("Int64")
    })
    .dropna(subset=["sido", "year"])  # sigungu가 없는 행은 남기고 싶으면 여기서 제외하지 말 것
    .assign(sigungu=lambda x: x["sigungu"].replace({"": pd.NA}))
    .drop_duplicates()
    .sort_values(["sido", "sigungu", "year"], na_position="last")
    .reset_index(drop=True)
)

# 저장
uniq.to_csv("unique_trade.csv", index=False, encoding="utf-8-sig")

uniq.head()


,sido,sigungu,year
0,강원특별자치도,강릉시,2008
1,강원특별자치도,강릉시,2009
2,강원특별자치도,강릉시,2010
3,강원특별자치도,강릉시,2011
4,강원특별자치도,강릉시,2012


In [13]:
import re
import pandas as pd
from pathlib import Path

# 파일 경로
path = Path("/Users/kjh/Documents/geostat3_e9/manufacturing_production_index.csv")

# 1) 데이터 로드
df = pd.read_csv(path)

# 2) 컬럼명 소문자/공백 정리
df.columns = df.columns.str.strip().str.lower()

# 3) 후보 컬럼 자동 매핑
ALIAS = {
    "sido":    ["sido", "시도", "광역시도", "province", "sido_nm", "sido_name", "sido_name_kor"],
    "sigungu": ["sigungu", "시군구", "sgg", "시군구명", "sigungu_nm", "sigungu_name"],
    "region":  ["region", "지역", "행정구역", "행정구역명", "시도_시군구"],
    "year":    ["year", "연도", "yyyy"],
    "date":    ["date", "날짜", "ym", "yyyymm", "연월"]
}

def pick(cols, in_df):
    for c in cols:
        if c in in_df.columns:
            return c
    return None

sido_col    = pick(ALIAS["sido"], df)
sigungu_col = pick(ALIAS["sigungu"], df)
region_col  = pick(ALIAS["region"], df)
year_col    = pick(ALIAS["year"], df)
date_col    = pick(ALIAS["date"], df)

# ---------- (A) region -> (sido, sigungu) 분리 ----------
# 공식/변경 포함 시도명 후보(긴 것 먼저 매칭)
SIDO_NAMES = [
    "세종특별자치시", "제주특별자치도", "강원특별자치도", "전북특별자치도",
    "서울특별시", "부산광역시", "대구광역시", "인천광역시", "광주광역시", "대전광역시", "울산광역시",
    "경기도", "강원도", "충청북도", "충청남도", "전라북도", "전라남도", "경상북도", "경상남도"
]
# 영문/축약 대응(있다면)
SIDO_EN_MAP = {
    "seoul": "서울특별시", "busan": "부산광역시", "daegu": "대구광역시", "incheon": "인천광역시",
    "gwangju": "광주광역시", "daejeon": "대전광역시", "ulsan": "울산광역시", "sejong": "세종특별자치시",
    "gyeonggi": "경기도", "gangwon": "강원특별자치도", "chungbuk": "충청북도", "chungnam": "충청남도",
    "jeonbuk": "전북특별자치도", "jeonnam": "전라남도", "gyeongbuk": "경상북도", "gyeongnam": "경상남도",
    "jeju": "제주특별자치도"
}

def clean_region(s):
    if pd.isna(s):
        return None
    s = str(s)
    s = re.sub(r"[\(\[\{].*?[\)\]\}]", "", s)  # 괄호 내용 제거
    s = s.replace("/", " ").replace("-", " ").replace(",", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_region(s):
    """ region 문자열에서 (시도, 시군구) 추출 """
    if not s:
        return (None, None)
    raw = clean_region(s)

    # 1) 쉼표/슬래시로 이미 구분된 케이스 처리 후 재시도
    for sep in [",", "/", "-"]:
        if sep in str(s):
            parts = [p.strip() for p in str(s).split(sep)]
            if len(parts) >= 2:
                return split_region(" ".join(parts))  # 공백 기반 재처리

    # 2) 시도명 prefix 매칭(가장 긴 시도명 우선)
    for sido_name in sorted(SIDO_NAMES, key=len, reverse=True):
        if raw.startswith(sido_name):
            rest = raw[len(sido_name):].strip()
            return (sido_name, rest if rest else None)

    # 3) 영문/로마자 시도명 시작하는 경우
    first_tok = raw.split(" ")[0].lower()
    if first_tok in SIDO_EN_MAP:
        sido_name = SIDO_EN_MAP[first_tok]
        rest = raw[len(first_tok):].strip()
        # 영문 토큰 제거를 위해 첫 토큰 제외
        rest = " ".join(raw.split(" ")[1:]).strip() or None
        return (sido_name, rest)

    # 4) fallback: 첫 토큰을 시도, 나머지를 시군구로
    toks = raw.split(" ")
    if len(toks) == 1:
        return (toks[0], None)
    return (toks[0], " ".join(toks[1:]))

# region이 있고(또는 시도/시군구 중 하나라도 없으면) region에서 파생
if region_col is not None and (sido_col is None or sigungu_col is None):
    tmp = df[region_col].map(clean_region)
    parsed = tmp.map(split_region)
    df["_sido_from_region"]    = parsed.map(lambda x: x[0])
    df["_sigungu_from_region"] = parsed.map(lambda x: x[1])
    # 우선순위: 기존 컬럼 > region 파생
    if sido_col is None:
        sido_col = "_sido_from_region"
    if sigungu_col is None:
        sigungu_col = "_sigungu_from_region"

# 최종 체크
if sido_col is None or sigungu_col is None:
    raise KeyError("시도/시군구 컬럼을 찾지 못했습니다. (region 분해 실패 가능)")

# ---------- (B) 연도 파생 ----------
if year_col is None:
    if date_col is None:
        raise KeyError("연도(year) 또는 연월(date/yyyymm) 관련 컬럼을 찾지 못했습니다.")
    year_series = pd.to_datetime(
        df[date_col].astype(str).str.replace(r"[^0-9]", "", regex=True).str[:6],
        format="%Y%m", errors="coerce"
    ).dt.year
else:
    year_series = pd.to_numeric(df[year_col], errors="coerce")
    if year_series.isna().all() and date_col is not None:
        year_series = pd.to_datetime(
            df[date_col].astype(str).str.replace(r"[^0-9]", "", regex=True).str[:6],
            format="%Y%m", errors="coerce"
        ).dt.year

# ---------- (C) 유일 조합 테이블 ----------
uniq = (
    pd.DataFrame({
        "sido": df[sido_col].astype(str).str.strip(),
        "sigungu": df[sigungu_col].astype(str).str.strip(),
        "year": year_series.astype("Int64")
    })
    .dropna(subset=["sido", "year"])  # sigungu가 없는 행은 남기고 싶으면 여기서 제외하지 말 것
    .assign(sigungu=lambda x: x["sigungu"].replace({"": pd.NA}))
    .drop_duplicates()
    .sort_values(["sido", "sigungu", "year"], na_position="last")
    .reset_index(drop=True)
)

# 저장
uniq.to_csv("unique_manufacturing.csv", index=False, encoding="utf-8-sig")

uniq.head()


,sido,sigungu,year
0,강원도,None,2008
1,강원도,None,2009
2,강원도,None,2010
3,강원도,None,2011
4,강원도,None,2012


In [30]:
# -*- coding: utf-8 -*-
"""
unique_pop 생성 (검증된 컬럼명 사용)
- 입력 : /Users/kjh/Documents/geostat3_e9/population_panel_sgg_aggregated.csv
- 출력 : unique_pop.csv  (컬럼: sido, sigungu, year)
- 비고 : 광역·특별(자치)시는 '시도명 + 구/군' 형태가 이미 들어 있으므로
        남구/서구/중구 등의 중복 문제 없이 안전.
"""

import pandas as pd
from pathlib import Path

# 경로 설정 (필요시 수정)
IN_PATH  = Path("/Users/kjh/Documents/geostat3_e9/population_panel_sgg_aggregated.csv")
OUT_PATH = Path("unique_pop.csv")  # 현재 작업 디렉토리에 저장

# 1) 로드
df = pd.read_csv(IN_PATH)

# 2) 필수 컬럼 존재 확인 (실제 파일 기준)
required = {"sido_name", "sigungu", "year"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"필수 컬럼 누락: {missing}  (파일 컬럼: {list(df.columns)})")

# 3) 정리 & 유일 조합
uniq = (
    df.loc[:, ["sido_name", "sigungu", "year"]]
      .rename(columns={"sido_name": "sido"})
      .assign(
          sido=lambda x: x["sido"].astype(str).str.strip(),
          sigungu=lambda x: x["sigungu"].astype(str).str.strip(),
          year=lambda x: pd.to_numeric(x["year"], errors="coerce").astype("Int64")
      )
      .dropna(subset=["sido", "sigungu", "year"])
      .drop_duplicates()
      .sort_values(["sido", "sigungu", "year"])
      .reset_index(drop=True)
)

# 4) 저장
uniq.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"[OK] unique_pop.csv saved: {OUT_PATH}  rows={len(uniq):,}")
print(uniq.head(10))


[OK] unique_pop.csv saved: unique_pop.csv  rows=4,211
      sido sigungu  year
0  강원특별자치도     강릉시  2008
1  강원특별자치도     강릉시  2009
2  강원특별자치도     강릉시  2010
3  강원특별자치도     강릉시  2011
4  강원특별자치도     강릉시  2012
5  강원특별자치도     강릉시  2013
6  강원특별자치도     강릉시  2014
7  강원특별자치도     강릉시  2015
8  강원특별자치도     강릉시  2016
9  강원특별자치도     강릉시  2017


## Construct Matching Table

In [31]:
# -*- coding: utf-8 -*-
"""
매칭 테이블 생성 (시군구 × 연-월 조인키)
- 코드북: adm_code_general_2024.xlsx (sheet='Transform')
- 소스: unique_e9.csv, unique_pop.csv, unique_trade.csv, (옵션) unique_manufacturing.csv

출력:
  - matching_table_join_keys.csv
  - unmatched_rows_<src>.csv

포인트
- 시도+시군구 복합키(join_key)로 1차 매칭
- Transform 시트의 대체명칭 열들을 이용해 code_key를 확장
- 강원/전북 특별자치도 등 시도 별칭 정규화
- (중요) 소스의 sigungu가 '서울특별시 강남구' 같은 형태면 시도 접두어 제거
- 소스/코드 모두 숫자 프리픽스('11000 서울특별시') 제거
"""

from pathlib import Path
import pandas as pd
import numpy as np
import unicodedata
import re

# -----------------------------
# 경로 & 소스
# -----------------------------
BASE = Path("/Users/kjh/Documents/geostat3_e9")
CODEBOOK_XLSX = BASE / "adm_code_general_2024.xlsx"
SOURCES = {
    "e9":   BASE / "unique_e9.csv",
    "pop":  BASE / "unique_pop.csv",
    "trade":BASE / "unique_trade.csv",
    # "manufacturing": BASE / "unique_manufacturing.csv",
}

# (옵션) 연도 필터
YEAR_START = 2009   # 예: 2009
YEAR_END   = None   # 예: 2025

# -----------------------------
# 유틸
# -----------------------------
def normalize_korean(text: str) -> str:
    """공백/괄호 제거 + NFC 정규화"""
    if pd.isna(text):
        return text
    s = unicodedata.normalize("NFC", str(text)).strip()
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[()（）\[\]]", "", s)
    return s

SIDO_ALIAS = {
    normalize_korean("강원도"):          normalize_korean("강원특별자치도"),
    normalize_korean("강원특별자치도"):  normalize_korean("강원특별자치도"),
    normalize_korean("전라북도"):        normalize_korean("전북특별자치도"),
    normalize_korean("전북특별자치도"):  normalize_korean("전북특별자치도"),
    normalize_korean("전북"):            normalize_korean("전북특별자치도"),
    normalize_korean("세종시"):          normalize_korean("세종특별자치시"),
    normalize_korean("세종특별자치시"):  normalize_korean("세종특별자치시"),
    normalize_korean("제주도"):          normalize_korean("제주특별자치도"),
    normalize_korean("제주특별자치도"):  normalize_korean("제주특별자치도"),
}

def canonicalize_sido(s: str) -> str:
    if pd.isna(s):
        return s
    s_norm = normalize_korean(s)
    return SIDO_ALIAS.get(s_norm, s_norm)

def clean_sido_name(x: str) -> str:
    """'001 서울특별시' → '서울특별시'"""
    if pd.isna(x):
        return x
    return re.sub(r"^\s*\d+\s*", "", str(x)).strip()

def digits(s):
    return re.sub(r"[^0-9]", "", str(s)) if pd.notna(s) else ""

def coerce_yyyymm(df: pd.DataFrame) -> pd.Series:
    """date/yyyymm/ym/연월 → 6자리 추출, 없으면 year+month, 없으면 year*100+01"""
    cols_lower = {c.lower(): c for c in df.columns}
    for k in ["date", "yyyymm", "ym", "연월", "year_month"]:
        if k in cols_lower:
            s = df[cols_lower[k]].astype(str).str.replace(r"[^0-9]", "", regex=True).str[:6]
            s = s.replace("", np.nan)
            return s.astype("Int64")
    ycol = next((cols_lower[k] for k in ["year", "연도", "yyyy", "yr"] if k in cols_lower), None)
    mcol = next((cols_lower[k] for k in ["month", "월", "mm", "mnth"] if k in cols_lower), None)
    if ycol and mcol:
        yy = pd.to_numeric(df[ycol], errors="coerce").astype("Int64")
        mm = pd.to_numeric(df[mcol], errors="coerce").astype("Int64")
        return (yy * 100 + mm).astype("Int64")
    if ycol:
        yy = pd.to_numeric(df[ycol], errors="coerce").astype("Int64")
        return (yy * 100 + 1).astype("Int64")
    raise ValueError("연-월(yyyymm) 정보를 유추할 수 없습니다.")

def strip_leading_digits(x: str) -> str:
    """앞 숫자 토큰 제거: '11000 서울특별시' → '서울특별시'"""
    if pd.isna(x):
        return x
    return re.sub(r"^\s*\d+\s*", "", str(x)).strip()

def strip_city_prefix(sigungu_norm: str, sido_norm: str) -> str:
    """
    '서울특별시강남구' → '강남구' (이미 normalize된 문자열 기준)
    시도명과 완전히 같은 경우(세종특별자치시 등)는 그대로 둠.
    """
    if pd.isna(sigungu_norm) or pd.isna(sido_norm):
        return sigungu_norm
    sgg = str(sigungu_norm); sido = str(sido_norm)
    if sgg == sido:
        return sgg
    return sgg[len(sido):] if sgg.startswith(sido) else sgg

# -----------------------------
# 코드북 Transform → code_key 확장
# -----------------------------
code = pd.read_excel(CODEBOOK_XLSX, sheet_name="Transform", dtype=str).rename(columns=str.strip)

# 기본 키(주명칭)
base = code[["SIDO","SGGNM_ADM","SGGCD_ADM_2024","SGIS_2019"]].copy()
base["sido_name"]    = base["SIDO"].map(clean_sido_name)
base["sido_norm"]    = base["sido_name"].map(normalize_korean).map(canonicalize_sido)
base["sigungu_name"] = base["SGGNM_ADM"].astype(str).map(strip_leading_digits)
base["sigungu_norm"] = base["sigungu_name"].map(normalize_korean)
base["sgg_code10"]   = base["SGGCD_ADM_2024"].map(digits)
base["sgis7_2019"]   = base["SGIS_2019"].map(digits)

# 대체 명칭을 키로 확장
ALT_SIG_COLS = [
    "SIGUNGU_NM","SIGUNGU","SIGUNGU_SUB_NM","SIGUNGU_PSEUDO","SIGUNGU_PSEUDO0",
    "SGGNM_DOJ","SGGNM_DCNM"
]
alts = []
for c in ALT_SIG_COLS:
    if c in code.columns:
        t = code[["SIDO", c, "SGGCD_ADM_2024", "SGIS_2019"]].copy()
        t = t.rename(columns={c: "alt_sigungu"})
        t["sido_name"]    = t["SIDO"].map(clean_sido_name)
        t["sido_norm"]    = t["sido_name"].map(normalize_korean).map(canonicalize_sido)
        t["sigungu_name"] = t["alt_sigungu"].astype(str).map(strip_leading_digits)
        t["sigungu_norm"] = t["sigungu_name"].map(normalize_korean)
        t["sgg_code10"]   = t["SGGCD_ADM_2024"].map(digits)
        t["sgis7_2019"]   = t["SGIS_2019"].map(digits)
        alts.append(t[["sido_norm","sigungu_norm","sido_name","sigungu_name","sgg_code10","sgis7_2019"]])

code_key = pd.concat(
    [base[["sido_norm","sigungu_norm","sido_name","sigungu_name","sgg_code10","sgis7_2019"]]] + alts,
    ignore_index=True
)
# 키 구성
code_key = code_key.dropna(subset=["sigungu_norm"])
code_key["join_key"] = (code_key["sido_norm"] + "::" + code_key["sigungu_norm"]).astype("string")
code_key = code_key.drop_duplicates(subset=["join_key"]).reset_index(drop=True)

# 전국에서 시군구명이 유일한 경우(보조 매칭용)
unique_sigungu = code_key.groupby("sigungu_norm")["sgg_code10"].nunique().rename("n_code").reset_index()
sigungu_unique_only = (
    code_key.merge(unique_sigungu, on="sigungu_norm", how="left")
            .query("n_code == 1")[["sigungu_norm","sgg_code10","sgis7_2019","sido_name","sigungu_name"]]
            .drop_duplicates()
)

# -----------------------------
# 소스 로드 & 매칭
# -----------------------------
def load_source(src_name: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)
    df.columns = df.columns.str.strip()

    if "sigungu" not in df.columns:
        raise ValueError(f"[{src_name}] 'sigungu' 컬럼이 필요합니다.")
    if "sido" not in df.columns:
        df["sido"] = np.nan

    # 숫자 프리픽스 제거(혼용 대비), 정규화, 시도 별칭, 시도 접두어 제거
    sido_raw    = df["sido"].map(strip_leading_digits).astype("string")
    sigungu_raw = df["sigungu"].map(strip_leading_digits).astype("string")

    out = pd.DataFrame({
        "sido_raw":    sido_raw,
        "sigungu_raw": sigungu_raw,
    })
    out["sido_norm"]    = out["sido_raw"].map(normalize_korean).map(canonicalize_sido).astype("string")
    out["sigungu_norm"] = out["sigungu_raw"].map(normalize_korean).astype("string")
    # 특별시/광역시 접두어 제거 (예: '서울특별시강남구' → '강남구')
    out["sigungu_norm"] = pd.Series(
        [strip_city_prefix(sgg, sido) for sgg, sido in zip(out["sigungu_norm"], out["sido_norm"])],
        dtype="string"
    )

    out["join_key"] = (out["sido_norm"] + "::" + out["sigungu_norm"]).astype("string")
    out["yyyymm"]   = coerce_yyyymm(df).astype("Int64")
    out["src"]      = src_name

    if YEAR_START is not None or YEAR_END is not None:
        yy = (out["yyyymm"] // 100).astype("Int64")
        if YEAR_START is not None:
            out = out[yy >= YEAR_START]
        if YEAR_END is not None:
            out = out[yy <= YEAR_END]

    return out.drop_duplicates(subset=["join_key","yyyymm"]).reset_index(drop=True)

loaded = {src: load_source(src, path) for src, path in SOURCES.items()}

matched_frames = {}
unmatched_frames = {}

for src, df in loaded.items():
    # 1차: 복합키로 정확 매칭
    m1 = df.merge(
        code_key[["join_key","sgg_code10","sgis7_2019","sido_name","sigungu_name"]],
        on="join_key", how="left"
    )

    # 2차: 1차 실패분 → 전국 유일 시군구명으로 보조 매칭
    need_fb = m1[m1["sgg_code10"].isna()].copy()
    if not need_fb.empty:
        fb = need_fb.merge(sigungu_unique_only, on="sigungu_norm", how="left", suffixes=("", "_fb"))
        for col in ["sgg_code10","sgis7_2019","sido_name","sigungu_name"]:
            m1.loc[fb.index, col] = m1.loc[fb.index, col].fillna(fb[col])

    matched   = m1[m1["sgg_code10"].notna()].copy()
    unmatched = m1[m1["sgg_code10"].isna()].copy()

    matched_frames[src] = matched[
        ["sgg_code10","sgis7_2019","yyyymm","sido_raw","sigungu_raw","sido_name","sigungu_name"]
    ].rename(columns={"sido_raw": f"sido_{src}", "sigungu_raw": f"sigungu_{src}"})

    unmatched_frames[src] = (
        unmatched[["yyyymm","sido_raw","sigungu_raw"]]
        .rename(columns={"sido_raw": "sido", "sigungu_raw": "sigungu"})
    )

# -----------------------------
# 마스터(와이드) 구성 & 저장
# -----------------------------
master = None
for src, mf in matched_frames.items():
    master = mf.copy() if master is None else master.merge(
        mf,
        on=["sgg_code10","sgis7_2019","yyyymm","sido_name","sigungu_name"],
        how="outer"
    )

for src in matched_frames.keys():
    master[f"present_{src}"] = master[f"sigungu_{src}"].notna().astype(int)

master = master.sort_values(["sgg_code10","yyyymm"]).reset_index(drop=True)

out_master = BASE / "matching_table_join_keys.csv"
master.to_csv(out_master, index=False, encoding="utf-8-sig")

for src, uf in unmatched_frames.items():
    out_unmatched = BASE / f"unmatched_rows_{src}.csv"
    (uf.sort_values(["yyyymm","sigungu"])
       .drop_duplicates()
       .to_csv(out_unmatched, index=False, encoding="utf-8-sig"))

print(f"[OK] code_key rows={len(code_key):,}, unique join_key={code_key['join_key'].nunique():,}")
print(f"[OK] Master saved: {out_master} rows={len(master):,} cols={len(master.columns)}")
for src, uf in unmatched_frames.items():
    print(f"[OK] Unmatched ({src}): {len(uf):,} -> {BASE / f'unmatched_rows_{src}.csv'}")


[OK] code_key rows=602, unique join_key=602
[OK] Master saved: /Users/kjh/Documents/geostat3_e9/matching_table_join_keys.csv rows=3,926 cols=14
[OK] Unmatched (e9): 21 -> /Users/kjh/Documents/geostat3_e9/unmatched_rows_e9.csv
[OK] Unmatched (pop): 82 -> /Users/kjh/Documents/geostat3_e9/unmatched_rows_pop.csv
[OK] Unmatched (trade): 53 -> /Users/kjh/Documents/geostat3_e9/unmatched_rows_trade.csv


### 특이값 처리

In [32]:
# -*- coding: utf-8 -*-
"""
[연-도 단위 커버리지 점검 + 공식명 열 + 전연도(17개) 미존재 목록]
- 파일: /Users/kjh/Documents/geostat3_e9/matching_table_join_keys.csv
- 코드북: /Users/kjh/Documents/geostat3_e9/adm_code_general_2024.xlsx (시트: '연계표_Data')

출력:
  1) missing_yearly_detail.csv   : (sgg_code10, year, missing_sources, sido_name, sigungu_name)
  2) missing_yearly_summary.csv  : (시군구별) 결측 연도 수, 커버리지율 + 공식명
  3) missing_all_years.csv       : 2009~2025 모든 연도에서 '3항목 모두 미존재' 시군구
  4) missing_all_years_long.csv  : (3)의 롱형(연도별 전부 missing 표기)

판정 기준:
- present_* 컬럼 3종(e9/pop/trade)이 같은 연도에 모두 1 → 그 연도는 "커버"
- 하나라도 0 → 그 연도는 "결측"
"""

from pathlib import Path
import pandas as pd
import numpy as np

# ---------------------- 설정 ----------------------
BASE = Path("/Users/kjh/Documents/geostat3_e9")
PATH_MT   = BASE / "matching_table_join_keys.csv"
PATH_CODE = BASE / "adm_code_general_2024.xlsx"

YEAR_START = 2009
YEAR_END   = 2025
TOTAL_YEARS = YEAR_END - YEAR_START + 1  # 17

# 파일 내 present_* 후보명(있으면 그 열 사용)
PRESENT_CANDIDATES = {
    "e9":    ["present_e9"],
    "pop":   ["present_pop", "present_population"],
    "trade": ["present_trade"],
}

def pick_present_column(df, candidates):
    """후보 리스트 중 실제 존재하는 첫 컬럼을 고른다. 없으면 None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

# ---------------------- 1) 매칭 테이블 로드 ----------------------
mt = pd.read_csv(PATH_MT, dtype=str).rename(columns=str.strip)

need_cols = {"sgg_code10", "yyyymm"}
missing_need = [c for c in need_cols if c not in mt.columns]
if missing_need:
    raise ValueError(f"필수 컬럼 누락: {missing_need}")

# present_* 만 남겨서 사용
present_cols_exist = [c for c in mt.columns if c.startswith("present_")]
mt = mt[["sgg_code10", "yyyymm"] + present_cols_exist].copy()

# 타입 정리
mt["sgg_code10"] = mt["sgg_code10"].astype(str)
mt["yyyymm"] = pd.to_numeric(mt["yyyymm"], errors="coerce").astype("Int64")
mt = mt.dropna(subset=["yyyymm"]).copy()
mt["year"] = (mt["yyyymm"] // 100).astype(int)

# 대상 연도 필터
mt = mt[(mt["year"] >= YEAR_START) & (mt["year"] <= YEAR_END)].copy()

# ---------------------- 2) present_* 3종 결정 ----------------------
present_cols = {}
for key, cand in PRESENT_CANDIDATES.items():
    sel = pick_present_column(mt, cand)
    if sel is not None:
        present_cols[key] = sel

expected = {"e9", "pop", "trade"}
if set(present_cols.keys()) != expected:
    raise ValueError(f"present_* 열 부족. 발견: {present_cols} / 기대: {expected}")

# 0/1 정수화
for k, col in present_cols.items():
    mt[col] = pd.to_numeric(mt[col], errors="coerce").fillna(0).astype(int)

check_cols = [present_cols["e9"], present_cols["pop"], present_cols["trade"]]

# ---------------------- 3) (sgg_code10, year)별 존재 여부 집계 ----------------------
yearly = (
    mt.groupby(["sgg_code10", "year"], as_index=False)[check_cols]
      .max()  # 같은 연도에 중복행이 있으면 '하나라도 1이면 1'
)
yearly["all_present"] = (yearly[check_cols].sum(axis=1) == 3).astype(int)

# --- 어떤 소스가 빠졌는지 플래그(0/1) 생성 ---
yearly["miss_e9"]   = (1 - yearly[present_cols["e9"]]).astype(int)
yearly["miss_pop"]  = (1 - yearly[present_cols["pop"]]).astype(int)
yearly["miss_trd"]  = (1 - yearly[present_cols["trade"]]).astype(int)

# --- 결측 상세 테이블: 실제로 빠진 소스만 문자열로 나열 ---
def _list_missing(row) -> str:
    items = []
    if row["miss_e9"]  == 1: items.append("e9")
    if row["miss_pop"] == 1: items.append("population")
    if row["miss_trd"] == 1: items.append("trade")
    return ",".join(items)

missing_detail = yearly[yearly["all_present"] == 0].copy()
missing_detail["missing_sources"] = missing_detail.apply(_list_missing, axis=1)

# 필요 컬럼만 남기고 정렬
missing_detail = (missing_detail[["sgg_code10", "year", "missing_sources"]]
                  .sort_values(["sgg_code10","year"])
                  .reset_index(drop=True))

# (커버리지) 시군구별 3항목 모두 충족한 연도 수
have = yearly.groupby("sgg_code10")["all_present"].sum().rename("covered_years")

# ---------------------- 4) 시군구별 요약 ----------------------
summary = (
    missing_detail.groupby("sgg_code10", as_index=False)["year"]
                  .agg(missing_years_count="nunique")
                  .merge(have, on="sgg_code10", how="right")  # 빠짐없는 시군구도 포함
                  .fillna({"missing_years_count": 0})
)
summary["missing_years_count"] = summary["missing_years_count"].astype(int)
summary["coverage_years"] = summary["covered_years"].astype(int)
summary["coverage_rate"] = (summary["coverage_years"] / TOTAL_YEARS * 100).round(1)

# ---------------------- 5) 코드북에서 공식명 붙이기 ----------------------
code = pd.read_excel(PATH_CODE, sheet_name="연계표_Data", dtype=str).rename(columns=str.strip)

sgg = code.loc[code["LEVEL"] == "시군구", ["ADM2", "KOR_NM"]].copy()
sgg = sgg.rename(columns={"ADM2": "sgg_code10", "KOR_NM": "sigungu_name"})
sgg["sgg_code10"] = sgg["sgg_code10"].astype(str)

sido = code.loc[code["LEVEL"] == "시도", ["ADM2", "KOR_NM"]].copy()
sido = sido.rename(columns={"ADM2": "sido_code10", "KOR_NM": "sido_name"})
sido["sido_code10"] = sido["sido_code10"].astype(str)

# 시군구 → 시도코드(앞 2자리 + '00000000')
sgg["sido_code10"] = sgg["sgg_code10"].str.slice(0, 2) + "00000000"
name_map = (
    sgg.merge(sido, on="sido_code10", how="left")
       [["sgg_code10", "sido_name", "sigungu_name"]]
       .drop_duplicates()
)

# 이름 붙이기
missing_detail = (
    missing_detail.merge(name_map, on="sgg_code10", how="left")
                  [["sgg_code10", "sido_name", "sigungu_name", "year", "missing_sources"]]
                  .sort_values(["sido_name", "sigungu_name", "year"])
)
summary = (
    summary.merge(name_map, on="sgg_code10", how="left")
           [["sgg_code10", "sido_name", "sigungu_name", "coverage_years", "missing_years_count", "coverage_rate"]]
           .sort_values(["sido_name", "sigungu_name"])
)

# ---------------------- 6) 저장(상세/요약) ----------------------
out_detail  = BASE / "missing_yearly_detail.csv"
out_summary = BASE / "missing_yearly_summary.csv"
missing_detail.to_csv(out_detail, index=False, encoding="utf-8-sig")
summary.to_csv(out_summary, index=False, encoding="utf-8-sig")

print(f"[OK] 연도 범위 {YEAR_START}~{YEAR_END} (총 {TOTAL_YEARS}년) 검증 완료")
print(f" - 결측 연도 있는 시군구 수: {(summary['missing_years_count'] > 0).sum()} / 전체 {summary.shape[0]}")
print(f" - 요약:  {out_summary}")
print(f" - 상세:  {out_detail}")

# ---------------------- 7) (추가) 17개 연도 '모두 미존재' 시군구 ----------------------
# 주의: 연도필터 이후에는 아예 사라져 누락될 수 있으므로, 필터 전 원본에서 우주집합을 만든다.
mt_full = pd.read_csv(PATH_MT, dtype=str).rename(columns=str.strip)
mt_full = mt_full[["sgg_code10", "yyyymm"]].dropna().drop_duplicates()
mt_full["sgg_code10"] = mt_full["sgg_code10"].astype(str)

# 관심 우주: 원자료에 등장한 모든 시군구
universe = mt_full[["sgg_code10"]].drop_duplicates()

# have(covered_years)와 좌결합 → 0년 커버 검출
summary_all = universe.merge(have.reset_index(), on="sgg_code10", how="left")
summary_all["covered_years"] = summary_all["covered_years"].fillna(0).astype(int)
summary_all["coverage_years"] = summary_all["covered_years"]
summary_all["coverage_rate"]  = (summary_all["coverage_years"] / TOTAL_YEARS * 100).round(1)
summary_all["missing_years_count"] = (TOTAL_YEARS - summary_all["coverage_years"]).astype(int)

# 공식명 붙여 정돈
summary_all = (
    summary_all.merge(name_map, on="sgg_code10", how="left")
               [["sgg_code10", "sido_name", "sigungu_name", "coverage_years", "missing_years_count", "coverage_rate"]]
)

# ★ 17개 연도 전부 미존재(=0년 커버)
zero_cov = summary_all[summary_all["coverage_years"] == 0].copy().sort_values(["sido_name", "sigungu_name"])

# 롱형 상세(2009~2025 전부 missing_sources 고정 기입)
years = pd.Series(range(YEAR_START, YEAR_END + 1), name="year")
zero_long = zero_cov[["sgg_code10", "sido_name", "sigungu_name"]].merge(years, how="cross")
zero_long["missing_sources"] = "e9,population,trade"

# 저장
out_zero_list = BASE / "missing_all_years.csv"
out_zero_long = BASE / "missing_all_years_long.csv"
zero_cov.to_csv(out_zero_list, index=False, encoding="utf-8-sig")
zero_long.to_csv(out_zero_long, index=False, encoding="utf-8-sig")

print(f"[OK] 17개 연도 전부 미존재 시군구: {len(zero_cov)}개 -> {out_zero_list}")
print(f"[OK] 17개 연도 전부 미존재 상세(long): {out_zero_long}")


[OK] 연도 범위 2009~2025 (총 17년) 검증 완료
 - 결측 연도 있는 시군구 수: 10 / 전체 235
 - 요약:  /Users/kjh/Documents/geostat3_e9/missing_yearly_summary.csv
 - 상세:  /Users/kjh/Documents/geostat3_e9/missing_yearly_detail.csv
[OK] 17개 연도 전부 미존재 시군구: 8개 -> /Users/kjh/Documents/geostat3_e9/missing_all_years.csv
[OK] 17개 연도 전부 미존재 상세(long): /Users/kjh/Documents/geostat3_e9/missing_all_years_long.csv


---

## Join

In [ ]:
# -*- coding: utf-8 -*-
"""
[패널+이름 열 추가] 시군구 × 연-월 패널 결합 (E9 + 인구 + 무역) + 행정구역 공식명 열
- 그리드: 각 소스의 실제 월별(yyyymm) 합집합
- sgg_code10 부여: matching_table_join_keys.csv 의 이름→코드 매핑 (소스별 실제 컬럼명)
- 최종 패널에 'sido_name', 'sigungu_name' (코드북 기준 공식명) 추가

경로 기준: /Users/kjh/Documents/geostat3_e9
입력:
  - matching_table_join_keys.csv
  - E9_panel_sgg_aggregated.csv              # cols: ['sido','sigungu','yyyymm','value','year','month']
  - population_panel_sgg_aggregated.csv      # cols: ['sido_name','sigungu','year','month','total','male','female','yyyymm']
  - trade_panel.csv                           # cols: ['time','region','export_cnt','export_val','import_cnt','import_val','trade_balance','year','month','file_year']
  - adm_code_general_2024.xlsx  (시트: '연계표_Data')
출력:
  - panel_sgg_monthly.csv
"""

from pathlib import Path
import pandas as pd
import numpy as np
import re, unicodedata

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE = Path("/Users/kjh/Documents/geostat3_e9")
PATH_MT    = BASE / "matching_table_join_keys.csv"
PATH_E9    = BASE / "E9_panel_sgg_aggregated.csv"
PATH_POP   = BASE / "population_panel_sgg_aggregated.csv"
PATH_TRAD  = BASE / "trade_panel.csv"
PATH_CODE  = BASE / "adm_code_general_2024.xlsx"
OUT_PATH   = BASE / "panel_sgg_monthly.csv"

# -------------------------------------------------------
# Utils (정규화만, 열 탐색/퍼지검색 없음)
# -------------------------------------------------------
def normalize_kr(s):
    if pd.isna(s): return s
    s = unicodedata.normalize("NFC", str(s)).strip()
    s = re.sub(r"\s+","", s)
    s = re.sub(r"[()（）\[\]]","", s)
    return s

def make_map_from_mt(mt: pd.DataFrame, sido_col: str, sigg_col: str) -> pd.DataFrame:
    """
    matching_table에서 (sido_col, sigg_col) → sgg_code10 매핑 테이블 생성
    - 실제 컬럼명만 사용 (광범위 탐색/alias 없음)
    - 동일 (시도,시군구)쌍에 여러 코드 있으면 최빈 코드 선택
    """
    if (sido_col not in mt.columns) or (sigg_col not in mt.columns):
        # 해당 소스 맵핑 정보가 없으면 빈 프레임 반환
        return pd.DataFrame(columns=["sido_norm","sigungu_norm","sgg_code10"]).drop_duplicates()

    tmp = mt[[sido_col, sigg_col, "sgg_code10"]].dropna()
    tmp["sido_norm"]    = tmp[sido_col].map(normalize_kr)
    tmp["sigungu_norm"] = tmp[sigg_col].map(normalize_kr)

    # (sido_norm, sigungu_norm) → 최빈 sgg_code10
    tmp = (tmp.groupby(["sido_norm","sigungu_norm"])["sgg_code10"]
              .agg(lambda s: s.value_counts().index[0])
              .reset_index())
    return tmp[["sido_norm","sigungu_norm","sgg_code10"]].drop_duplicates()

# -------------------------------------------------------
# Load matching table (소스별 이름→코드)
# -------------------------------------------------------
mt = pd.read_csv(PATH_MT, dtype=str).rename(columns=str.strip)

# 각 소스별 매핑 (matching_table의 실제 열 이름만 사용)
map_e9    = make_map_from_mt(mt,  "sido_e9",   "sigungu_e9")       if "sido_e9"   in mt.columns else pd.DataFrame(columns=["sido_norm","sigungu_norm","sgg_code10"])
map_pop   = make_map_from_mt(mt,  "sido_pop",  "sigungu_pop")      if "sido_pop"  in mt.columns else pd.DataFrame(columns=["sido_norm","sigungu_norm","sgg_code10"])
map_trade = make_map_from_mt(mt,  "sido_trade","sigungu_trade")    if "sido_trade" in mt.columns else pd.DataFrame(columns=["sido_norm","sigungu_norm","sgg_code10"])

# -------------------------------------------------------
# 1) E9 (실제 컬럼명 사용)
#     ['sido','sigungu','yyyymm','value','year','month']
# -------------------------------------------------------
e9 = pd.read_csv(PATH_E9, dtype=str)
e9["yyyymm"] = e9["yyyymm"].astype(int)
e9["sido_norm"]    = e9["sido"].map(normalize_kr)
e9["sigungu_norm"] = e9["sigungu"].map(normalize_kr)
e9["value"]        = pd.to_numeric(e9["value"], errors="coerce").fillna(0)

e9m = (e9.merge(map_e9, on=["sido_norm","sigungu_norm"], how="inner")
         .groupby(["sgg_code10","yyyymm"], as_index=False)["value"].sum()
         .rename(columns={"value":"e9"}))

# -------------------------------------------------------
# 2) Population (실제 컬럼명 사용)
#     ['sido_name','sigungu','year','month','total','male','female','yyyymm']
# -------------------------------------------------------
pop = pd.read_csv(PATH_POP, dtype=str)
# yyyymm 구성 (이미 있음)
pop["yyyymm"] = pop["yyyymm"].astype(int)
pop["sido_norm"]    = pop["sido_name"].map(normalize_kr)
pop["sigungu_norm"] = pop["sigungu"].map(normalize_kr)

for c in ["total","male","female"]:
    pop[c] = pd.to_numeric(pop[c], errors="coerce").fillna(0)

pop_agg = (pop.groupby(["sido_norm","sigungu_norm","yyyymm"], as_index=False)[["total","male","female"]]
              .sum())
popm = (pop_agg.merge(map_pop, on=["sido_norm","sigungu_norm"], how="inner")
             .groupby(["sgg_code10","yyyymm"], as_index=False)[["total","male","female"]]
             .sum()
             .rename(columns={"total":"population_total",
                              "male":"population_male",
                              "female":"population_female"}))

# -------------------------------------------------------
# 3) Trade (실제 컬럼명 사용)
#     ['time','region','export_cnt','export_val','import_cnt','import_val','trade_balance','year','month','file_year']
#     - region: "시도 시군구" 한 칸 공백 구분
# -------------------------------------------------------
tr = pd.read_csv(PATH_TRAD, dtype=str)

# region → sido/sigungu 분리 (최초 공백 기준 2분할)
split = tr["region"].str.split(r"\s+", n=1, expand=True)
tr["sido"]    = split[0]
tr["sigungu"] = split[1]

# yyyymm 구성
tr["year"]   = tr["year"].astype(int)
tr["month"]  = tr["month"].astype(int)
tr["yyyymm"] = tr["year"]*100 + tr["month"]

# 정규화 및 수치형 변환
tr["sido_norm"]    = tr["sido"].map(normalize_kr)
tr["sigungu_norm"] = tr["sigungu"].map(normalize_kr)
for col in ["export_cnt","export_val","import_cnt","import_val","trade_balance"]:
    tr[col] = pd.to_numeric(tr[col], errors="coerce").fillna(0)

tr_agg = (tr.groupby(["sido_norm","sigungu_norm","yyyymm"], as_index=False)
            [["export_cnt","export_val","import_cnt","import_val","trade_balance"]]
            .sum())

trm = (tr_agg.merge(map_trade, on=["sido_norm","sigungu_norm"], how="inner")
           .groupby(["sgg_code10","yyyymm"], as_index=False)
           [["export_cnt","export_val","import_cnt","import_val","trade_balance"]]
           .sum()
           .rename(columns={"export_cnt":"trade_export_cnt",
                            "export_val":"trade_export_val",
                            "import_cnt":"trade_import_cnt",
                            "import_val":"trade_import_val"}))

# -------------------------------------------------------
# 4) 패널 결합 (월 합집합 outer)
# -------------------------------------------------------
panel = e9m.merge(popm, on=["sgg_code10","yyyymm"], how="outer")
panel = panel.merge(trm, on=["sgg_code10","yyyymm"], how="outer")

# -------------------------------------------------------
# 5) 코드북 공식명 추가 (LEVEL=시군구/시도, 실제 컬럼명만 사용)
#     - 시군구: ['ADM2','KOR_NM']
#     - 시도:   ['ADM2','KOR_NM']  (시도코드 = 시군구코드 앞 2자리 + '00000000')
# -------------------------------------------------------
def build_name_map_from_transform(xlsx_path: Path) -> pd.DataFrame:
    """
    Transform 시트에서 sgg_code10 -> (sido_name, sigungu_name) 맵 생성.
    컬럼명이 다를 수 있어 보수적 탐색을 함.
    """
    import pandas as _pd
    tr = _pd.read_excel(xlsx_path, sheet_name="Transform", dtype=str)
    tr.columns = tr.columns.str.strip()

    # 후보 키/이름 컬럼 잡기
    cand_code = [c for c in tr.columns if c.lower() in {"adm2", "sgg_code10", "sggcode10"}]
    cand_sgg  = [c for c in tr.columns if "sigungu" in c.lower() or "sgg" in c.lower() or "군구" in c]
    cand_sido = [c for c in tr.columns if "sido" in c.lower() or "시도" in c]

    if not cand_code or not cand_sgg or not cand_sido:
        raise ValueError("Transform 시트에서 필요한 열(ADM2/시군구/시도)을 찾지 못함")

    code_col = cand_code[0]
    sgg_col  = cand_sgg[0]
    sido_col = cand_sido[0]

    nm = tr[[code_col, sido_col, sgg_col]].dropna()
    nm = nm.rename(columns={code_col:"sgg_code10", sido_col:"sido_name", sgg_col:"sigungu_name"})
    nm["sgg_code10"] = nm["sgg_code10"].astype(str)
    return nm[["sgg_code10","sido_name","sigungu_name"]].drop_duplicates()

def build_name_map_from_link(xlsx_path: Path) -> pd.DataFrame:
    """
    연계표_Data 시트(LEVEL=시군구/시도)에서 맵 생성
    """
    import pandas as _pd
    code = _pd.read_excel(xlsx_path, sheet_name="연계표_Data", dtype=str).rename(columns=str.strip)
    sgg = code.loc[code["LEVEL"]=="시군구", ["ADM2","KOR_NM"]].copy()
    sgg = sgg.rename(columns={"ADM2":"sgg_code10", "KOR_NM":"sigungu_name"})
    sgg["sgg_code10"] = sgg["sgg_code10"].astype(str)

    sido = code.loc[code["LEVEL"]=="시도", ["ADM2","KOR_NM"]].copy()
    sido = sido.rename(columns={"ADM2":"sido_code10", "KOR_NM":"sido_name"})
    sido["sido_code10"] = sido["sido_code10"].astype(str)

    sgg["sido_code10"] = sgg["sgg_code10"].str.slice(0,2) + "00000000"
    nm = sgg.merge(sido, on="sido_code10", how="left")[["sgg_code10","sido_name","sigungu_name"]].drop_duplicates()
    return nm

# 5-1) 공식 이름 맵 구성 (Transform 우선)
try:
    name_map = build_name_map_from_transform(PATH_CODE)
except Exception:
    name_map = build_name_map_from_link(PATH_CODE)

# 5-2) 패널에 공식명 붙이기
panel = panel.merge(name_map, on="sgg_code10", how="left")

# 5-3) 여전히 비면 매칭테이블의 소스 이름으로 보정(공식명 비어있는 행만)
#      우선순위: pop → e9 → trade
src_name_cols_priority = [
    ("sido_pop", "sigungu_pop"),
    ("sido_e9", "sigungu_e9"),
    ("sido_trade", "sigungu_trade"),
]
if panel["sido_name"].isna().any() or panel["sigungu_name"].isna().any():
    mt_cols = set(mt.columns)
    # mt에서 sgg_code10 단위로 '가장 많이 등장한 이름'을 대표로 사용
    for cs, cg in src_name_cols_priority:
        if cs in mt_cols and cg in mt_cols:
            tmp = mt[[ "sgg_code10", cs, cg ]].dropna()
            if not tmp.empty:
                # sgg_code10별 최빈값 선택
                rep = (tmp.groupby("sgg_code10")[[cs, cg]]
                          .agg(lambda x: x.value_counts().index[0])
                          .reset_index()
                          .rename(columns={cs:"sido_name_src", cg:"sigungu_name_src"}))
                panel = panel.merge(rep, on="sgg_code10", how="left")
                # 공식명이 비어있는 곳만 소스명으로 채움
                panel["sido_name"]    = panel["sido_name"].fillna(panel["sido_name_src"])
                panel["sigungu_name"] = panel["sigungu_name"].fillna(panel["sigungu_name_src"])
                panel = panel.drop(columns=["sido_name_src","sigungu_name_src"], errors="ignore")

# -------------------------------------------------------
# 6) year/month 추가, 정렬, 저장
# -------------------------------------------------------
panel["yyyymm"] = panel["yyyymm"].astype("Int64")
panel["year"]   = (panel["yyyymm"] // 100).astype("Int64")
panel["month"]  = (panel["yyyymm"] % 100).astype("Int64")

# 2009년 이후(포함)만 남기기
panel = panel[panel["yyyymm"] >= 200901].copy()

# 정렬·저장
panel = panel.sort_values(["sgg_code10","yyyymm"]).reset_index(drop=True)
panel.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"[OK] saved: {OUT_PATH} | shape={panel.shape}")
print(panel.head(10)[[
    "sgg_code10","sido_name","sigungu_name","yyyymm",
    "e9","population_total","population_male","population_female",
    "trade_export_cnt","trade_export_val","trade_import_cnt","trade_import_val","trade_balance"
]])


[OK] saved: /Users/kjh/Documents/geostat3_e9/panel_sgg_monthly.csv | shape=(48819, 15)
  sgg_code10 sido_name sigungu_name  yyyymm  e9  population_total  \
0      11110     서울특별시          종로구  200801 NaN               NaN   
1      11110     서울특별시          종로구  200802 NaN               NaN   
2      11110     서울특별시          종로구  200803 NaN               NaN   
3      11110     서울특별시          종로구  200804 NaN               NaN   
4      11110     서울특별시          종로구  200805 NaN               NaN   
5      11110     서울특별시          종로구  200806 NaN               NaN   
6      11110     서울특별시          종로구  200807 NaN               NaN   
7      11110     서울특별시          종로구  200808 NaN               NaN   
8      11110     서울특별시          종로구  200809 NaN               NaN   
9      11110     서울특별시          종로구  200810 NaN               NaN   

   population_male  population_female  trade_export_cnt  trade_export_val  \
0              NaN                NaN            2946.0           51227.0   